In [1]:
from src import *
from myutils import *
import gc

In [2]:
# def LoadEstimates_legacy(file):
#     file = os.path.expanduser(file)
#     npz = np.load(file, allow_pickle=True)
#     meta = MetaData(**npz['metadata'].item().__dict__, estimating='frequency')
#     c = Estimates(cropped_data=np.round(npz['cropped_data']), background=np.round(npz['noise']), metadata=meta)
#     return c

In [3]:
tree = [
    'noisy/di_5px',
    'noisy/spade_5px',

    '0noise/di_5px',
    '0noise/spade_5px',

    '0noise/di_3px',
    '0noise/spade_3px',

    '0noise_sign/di_5px',
    '0noise_sign/spade_5px',

    '0noise_sign/di_3px',
    '0noise_sign/spade_3px',
]

In [4]:
for f in ('__estimates__', '__simulations__'):
    @parallelize()
    def main(_folder):
        src = os.path.join(f'{f}.old', _folder)
        dst = os.path.join(f'{f}', _folder)
        for file in tqdm(glob(src+'/*')):
            c = LoadEstimates(file)
            c.metadata.methods = ('mle', 'lse') if c.metadata.measurement.upper() == 'DI' else ('mle', 'lse')
            try:
                c_ = Estimation.FromEstimates(c)
                if not os.path.exists(dst):
                    os.makedirs(dst)
                c_.savez(dst)
            except (ValueError, RuntimeError):
                print(f'{file} error')

            gc.collect()

    main(tree)